In [1]:
from pathlib import Path
import json
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
headers_json_path = Path('../headers/headers.json')
headers_zip_path = Path('../headers/headers.zip')
if headers_json_path.exists():
    with headers_json_path.open() as handle:
        headers = json.load(handle)
elif headers_zip_path.exists():
    with zipfile.ZipFile(headers_zip_path) as archive:
        with archive.open('headers.json') as handle:
            headers = json.load(handle)
else:
    raise FileNotFoundError('Expected ../headers/headers.json or ../headers/headers.zip')
df = pd.DataFrame(headers)
df

,hash,bits,time
0,000000000019d6689c085ae165831e934ff763ae46a2a6...,1d00ffff,1231006505
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1d00ffff,1231469665
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1d00ffff,1231469744
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1d00ffff,1231470173
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1d00ffff,1231470988
...,...,...,...
865038,00000000000000000002af876a1d8ce11e1c07018b016a...,17030ecd,1728572242
865039,00000000000000000000445bb08413ba56b56068395864...,17030ecd,1728572873
865040,000000000000000000019ca8e0127b00104b98ef7297e4...,17030ecd,1728573656
865041,0000000000000000000116551169c5d38e6b66b8fe27ab...,17030ecd,1728574093


In [3]:
def bits_to_target(bits):
    """Converts bits to target."""
    if isinstance(bits, str):
        bits = int(bits, 16)
    bits_n = (bits >> 24) & 0xff
    bits_base = bits & 0xffffff
    target = bits_base << (8 * (bits_n - 3))
    return target

GENESIS_TARGET = bits_to_target('1d00ffff')

df = df.copy()
df['target'] = df['bits'].apply(bits_to_target)
df['difficulty'] = GENESIS_TARGET / df['target']
df

,hash,bits,time,target,difficulty
0,000000000019d6689c085ae165831e934ff763ae46a2a6...,1d00ffff,1231006505,2695953529101130949315647634472399133601089873...,1.0
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1d00ffff,1231469665,2695953529101130949315647634472399133601089873...,1.0
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1d00ffff,1231469744,2695953529101130949315647634472399133601089873...,1.0
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1d00ffff,1231470173,2695953529101130949315647634472399133601089873...,1.0
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1d00ffff,1231470988,2695953529101130949315647634472399133601089873...,1.0
...,...,...,...,...,...
865038,00000000000000000002af876a1d8ce11e1c07018b016a...,17030ecd,1728572242,2928805436162009520992638294218449682899899138...,92049594548485.46875
865039,00000000000000000000445bb08413ba56b56068395864...,17030ecd,1728572873,2928805436162009520992638294218449682899899138...,92049594548485.46875
865040,000000000000000000019ca8e0127b00104b98ef7297e4...,17030ecd,1728573656,2928805436162009520992638294218449682899899138...,92049594548485.46875
865041,0000000000000000000116551169c5d38e6b66b8fe27ab...,17030ecd,1728574093,2928805436162009520992638294218449682899899138...,92049594548485.46875


In [4]:
df = df.copy()
if not np.issubdtype(df['time'].dtype, np.number):
    raise TypeError('Column time must be numeric (Unix seconds).')
df['datetime'] = pd.to_datetime(df['time'], unit='s', utc=True)
df

,hash,bits,time,target,difficulty,datetime
0,000000000019d6689c085ae165831e934ff763ae46a2a6...,1d00ffff,1231006505,2695953529101130949315647634472399133601089873...,1.0,2009-01-03 18:15:05+00:00
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1d00ffff,1231469665,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:54:25+00:00
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1d00ffff,1231469744,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:55:44+00:00
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1d00ffff,1231470173,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:02:53+00:00
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1d00ffff,1231470988,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:16:28+00:00
...,...,...,...,...,...,...
865038,00000000000000000002af876a1d8ce11e1c07018b016a...,17030ecd,1728572242,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 14:57:22+00:00
865039,00000000000000000000445bb08413ba56b56068395864...,17030ecd,1728572873,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:07:53+00:00
865040,000000000000000000019ca8e0127b00104b98ef7297e4...,17030ecd,1728573656,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:20:56+00:00
865041,0000000000000000000116551169c5d38e6b66b8fe27ab...,17030ecd,1728574093,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:28:13+00:00


In [5]:
df.dtypes

hash                         str
bits                         str
time                       int64
target                    object
difficulty                object
datetime      datetime64[s, UTC]
dtype: object

In [6]:
df = df.copy()
df['delta_t'] = df['time'].diff()
df

,hash,bits,time,target,difficulty,datetime,delta_t
0,000000000019d6689c085ae165831e934ff763ae46a2a6...,1d00ffff,1231006505,2695953529101130949315647634472399133601089873...,1.0,2009-01-03 18:15:05+00:00,NaN
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1d00ffff,1231469665,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:54:25+00:00,463160.0
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1d00ffff,1231469744,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:55:44+00:00,79.0
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1d00ffff,1231470173,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:02:53+00:00,429.0
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1d00ffff,1231470988,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:16:28+00:00,815.0
...,...,...,...,...,...,...,...
865038,00000000000000000002af876a1d8ce11e1c07018b016a...,17030ecd,1728572242,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 14:57:22+00:00,64.0
865039,00000000000000000000445bb08413ba56b56068395864...,17030ecd,1728572873,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:07:53+00:00,631.0
865040,000000000000000000019ca8e0127b00104b98ef7297e4...,17030ecd,1728573656,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:20:56+00:00,783.0
865041,0000000000000000000116551169c5d38e6b66b8fe27ab...,17030ecd,1728574093,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:28:13+00:00,437.0


In [7]:
invalid_delta_t = int((df['delta_t'] <= 0).sum())
print(f'Nombre de delta_t <= 0: {invalid_delta_t}')

df_valid = df[df['delta_t'] > 0].copy()
print(f'Lignes conservees apres filtre: {len(df_valid)} sur {len(df)}')

df_valid

Nombre de delta_t <= 0: 15111
Lignes conservees apres filtre: 849931 sur 865043


,hash,bits,time,target,difficulty,datetime,delta_t
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1d00ffff,1231469665,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:54:25+00:00,463160.0
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1d00ffff,1231469744,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 02:55:44+00:00,79.0
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1d00ffff,1231470173,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:02:53+00:00,429.0
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1d00ffff,1231470988,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:16:28+00:00,815.0
5,000000009b7262315dbf071787ad3656097b892abffd1f...,1d00ffff,1231471428,2695953529101130949315647634472399133601089873...,1.0,2009-01-09 03:23:48+00:00,440.0
...,...,...,...,...,...,...,...
865038,00000000000000000002af876a1d8ce11e1c07018b016a...,17030ecd,1728572242,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 14:57:22+00:00,64.0
865039,00000000000000000000445bb08413ba56b56068395864...,17030ecd,1728572873,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:07:53+00:00,631.0
865040,000000000000000000019ca8e0127b00104b98ef7297e4...,17030ecd,1728573656,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:20:56+00:00,783.0
865041,0000000000000000000116551169c5d38e6b66b8fe27ab...,17030ecd,1728574093,2928805436162009520992638294218449682899899138...,92049594548485.46875,2024-10-10 15:28:13+00:00,437.0


### Pourquoi cette formule de hashrate?

On utilise deux versions du même signal:
- `hashrate_relative = difficulty / delta_t` (proxy relatif, sans unité physique)
- `hashrate_hs_exact = difficulty * C_exact / delta_t` (estimation absolue en hashes/s)

Dérivation rapide:
- Probabilité de succès par hash: `p = target / 2^256`
- Nombre moyen de hashes pour trouver un bloc: `E[N] = 1/p = 2^256/target`
- Définition Bitcoin: `difficulty = GENESIS_TARGET / target`
- Donc `E[N] = difficulty * (2^256/GENESIS_TARGET)`

La constante exacte est:
`C_exact = 2^256 / GENESIS_TARGET`

Donc la version absolue rigoureuse (hashes/s) est:
`hashrate_hs_exact = difficulty * C_exact / delta_t`

Pourquoi on voit souvent `2^32`?
- `2^32` est une approximation conventionnelle très proche de `C_exact`
- Elle est suffisante pour des comparaisons relatives (ex. `gamma`)
- Pour une estimation absolue de hashrate, on préfère `C_exact`

Limite pratique: le signal bloc-par-bloc est bruité (Poisson + timestamps imparfaits), donc on le lissera ensuite.

In [8]:
C_EXACT = (2**256) / GENESIS_TARGET
df_valid['hashrate_relative'] = df_valid['difficulty'] / df_valid['delta_t']
df_valid['hashrate_hs_exact'] = df_valid['difficulty'] * C_EXACT / df_valid['delta_t']
df_valid[['hash','time','delta_t','difficulty','hashrate_relative','hashrate_hs_exact']].head()

,hash,time,delta_t,difficulty,hashrate_relative,hashrate_hs_exact
1,00000000839a8e6886ab5951d76f411475428afc90947e...,1231469665,463160.0,1.0,0.000002,9273.324193
2,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,1231469744,79.0,1.0,0.012658,54367504.21519
3,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,1231470173,429.0,1.0,0.002331,10011731.545455
4,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,1231470988,815.0,1.0,0.001227,5269978.936196
5,000000009b7262315dbf071787ad3656097b892abffd1f...,1231471428,440.0,1.0,0.002273,9761438.256818


In [11]:
S_MAX = 50
series_rel = df_valid['hashrate_relative'].dropna().reset_index(drop=True)
series_abs = df_valid['hashrate_hs_exact'].dropna().reset_index(drop=True)
gamma_s_rel = -np.inf
gamma_s_abs = -np.inf
best_rel = {'window_size': None, 'end_idx': None}
best_abs = {'window_size': None, 'end_idx': None}
for w in range(2, S_MAX + 1):
    roll_max_rel = series_rel.rolling(window=w, min_periods=w).max()
    roll_min_rel = series_rel.rolling(window=w, min_periods=w).min()
    ratio_rel = (roll_max_rel / roll_min_rel).replace([np.inf, -np.inf], np.nan)
    local_rel = ratio_rel.max()
    if pd.notna(local_rel) and local_rel > gamma_s_rel:
        gamma_s_rel = float(local_rel)
        best_rel['window_size'] = w
        best_rel['end_idx'] = int(ratio_rel.idxmax())
    roll_max_abs = series_abs.rolling(window=w, min_periods=w).max()
    roll_min_abs = series_abs.rolling(window=w, min_periods=w).min()
    ratio_abs = (roll_max_abs / roll_min_abs).replace([np.inf, -np.inf], np.nan)
    local_abs = ratio_abs.max()
    if pd.notna(local_abs) and local_abs > gamma_s_abs:
        gamma_s_abs = float(local_abs)
        best_abs['window_size'] = w
        best_abs['end_idx'] = int(ratio_abs.idxmax())
pd.DataFrame([{'metric':f'gamma_s_relative (|S| <= {S_MAX})','value':gamma_s_rel,'worst_window_size':best_rel['window_size'],'worst_window_end_idx':best_rel['end_idx']},{'metric':f'gamma_s_abs_exact (|S| <= {S_MAX})','value':gamma_s_abs,'worst_window_size':best_abs['window_size'],'worst_window_end_idx':best_abs['end_idx']}])

,metric,value,worst_window_size,worst_window_end_idx
0,gamma_s_relative (|S| <= 50),38596.666667,16,15
1,gamma_s_abs_exact (|S| <= 50),38596.666667,16,15


### Gamma conforme a la definition (gamma, s)-respecting

Ici on calcule `gamma_s` comme:
- max sur toutes les sous-fenetres contigues de taille `|S| <= s`
- du ratio `max(H)/min(H)` a l'interieur de chaque fenetre

`S_MAX` joue le role de `s` dans la definition formelle.